# Tutorial 2: Bring Your Own Dataset

This tutorial shows how to prepare any EHR dataset for OneEHR.

## OneEHR Data Model

OneEHR expects three simple CSV files:

| File | Columns | Description |
|------|---------|-------------|
| `dynamic.csv` | `patient_id`, `event_time`, `code`, `value` | Longitudinal events |
| `static.csv` | `patient_id`, ... | Patient demographics |
| `label.csv` | `patient_id`, `label_time`, `label_code`, `label_value` | Task labels |

In [ ]:
import pandas as pd
import numpy as np

# Create a synthetic dataset
rng = np.random.default_rng(42)
n_patients = 200

# Dynamic events: lab results + vitals over 5 days
rows = []
for pid in range(n_patients):
    for day in range(5):
        rows.append({
            "patient_id": f"P{pid:04d}",
            "event_time": f"2024-01-0{day+1}",
            "code": "heart_rate",
            "value": round(rng.normal(80, 15), 1),
        })
        rows.append({
            "patient_id": f"P{pid:04d}",
            "event_time": f"2024-01-0{day+1}",
            "code": "systolic_bp",
            "value": round(rng.normal(120, 20), 1),
        })
        rows.append({
            "patient_id": f"P{pid:04d}",
            "event_time": f"2024-01-0{day+1}",
            "code": "glucose",
            "value": round(rng.normal(100, 25), 1),
        })

dynamic = pd.DataFrame(rows)
dynamic.to_csv("my_dynamic.csv", index=False)
print(f"Dynamic: {len(dynamic)} events, {dynamic['patient_id'].nunique()} patients")
dynamic.head()

In [ ]:
# Static features: demographics
static = pd.DataFrame({
    "patient_id": [f"P{i:04d}" for i in range(n_patients)],
    "age": rng.integers(20, 90, size=n_patients),
    "sex": rng.choice(["M", "F"], size=n_patients),
})
static.to_csv("my_static.csv", index=False)
static.head()

In [ ]:
# Labels: mortality prediction
labels = pd.DataFrame({
    "patient_id": [f"P{i:04d}" for i in range(n_patients)],
    "label_time": "2024-01-05",
    "label_code": "mortality",
    "label_value": rng.integers(0, 2, size=n_patients),
})
labels.to_csv("my_label.csv", index=False)
print(f"Label distribution: {labels['label_value'].value_counts().to_dict()}")

## Using Built-in Dataset Converters

For standard datasets, use OneEHR's built-in converters:

In [ ]:
# Example: MIMIC-III conversion (requires data access)
# from oneehr.datasets import MIMIC3Converter
# converter = MIMIC3Converter("/path/to/mimic3")
# converter.save("mimic3_oneehr/", task="mortality")

# Or via CLI:
# !oneehr convert --dataset mimic3 --raw-dir /path/to/mimic3 --output-dir mimic3_oneehr/ --task mortality

print("Available converters: MIMIC3Converter, MIMIC4Converter, EICUConverter")

## Using Medical Code Ontologies

OneEHR provides code mapping utilities to aggregate ICD codes by chapter or CCS category:

In [ ]:
from oneehr.medcode import ICD9, ICD10, CodeMapper

# ICD code utilities
print(f"ICD-9 401.9 -> Chapter: {ICD9.chapter('401.9')}")
print(f"ICD-9 401.9 -> Category: {ICD9.category('401.9')}")
print(f"ICD-10 I10 -> Chapter: {ICD10.chapter('I10')}")

# Code mapping for dimensionality reduction
mapper = CodeMapper()
mapper.add_icd_chapter_mapping(version=9, prefix="DX_")
print(f"\nDX_4019 -> {mapper.map_code('DX_4019')}")
print(f"LAB_50801 -> {mapper.map_code('LAB_50801')}  (unchanged)")

## Next Steps

Once your data is in the three-table format, create a TOML config and run the full pipeline:

```bash
oneehr preprocess --config my_experiment.toml
oneehr train --config my_experiment.toml
oneehr test --config my_experiment.toml
oneehr analyze --config my_experiment.toml
```